# DAPR Pub/Sub Messaging Demo

This notebook demonstrates the `messaging` library using **DAPR** as transport.
Same `MessageBus` API, same `@bus.handler()` decorators, same `CloudEvent` model —
only the transport changes.

```
Notebook → MessageBus → DAPR Sidecar → Azure Service Bus → DAPR Sidecar → MessageBus → Handler
```

The CloudEvent on the wire is **identical** whether published via KNative or DAPR.
Run this side-by-side with `messaging-demo.ipynb` to compare.

## 1. Setup — Create a MessageBus with DAPR Transport

In [ ]:
import os
from messaging import MessageBus, CloudEvent, Disposition, MessageContext
from messaging.dapr import DaprTransport, DaprSubscription

DAPR_HTTP_PORT = int(os.environ.get("DAPR_HTTP_PORT", "3500"))
PUBSUB_NAME = os.environ.get("PUBSUB_NAME", "messaging")
PUBLISH_TOPIC = os.environ.get("PUBLISH_TOPIC", "knative-outbound")
SUBSCRIBE_TOPIC = os.environ.get("SUBSCRIBE_TOPIC", "knative-inbound")

bus = MessageBus()
bus.configure(DaprTransport(
    pubsub_name=PUBSUB_NAME,
    topic=PUBLISH_TOPIC,
    subscriptions=[DaprSubscription(PUBSUB_NAME, SUBSCRIBE_TOPIC)],
    dapr_http_port=DAPR_HTTP_PORT,
))

print(f"✅ MessageBus configured with DAPR transport")
print(f"   Pub/Sub component: {PUBSUB_NAME}")
print(f"   Publish topic:     {PUBLISH_TOPIC}")
print(f"   Subscribe topic:   {SUBSCRIBE_TOPIC}")
print(f"   DAPR sidecar:      http://localhost:{DAPR_HTTP_PORT}")

## 2. Publish a CloudEvent via DAPR

The event is serialized as a **structured CloudEvent** (full JSON with `specversion`,
`type`, `source`, etc.) and sent to the DAPR sidecar with `rawPayload=true`.

This means the CloudEvent arrives **as-is** on Azure Service Bus — no DAPR wrapper.

In [ ]:
event = CloudEvent(
    type="com.example.demo",
    source="/demo/jupyter-dapr",
    data={"message": "Hello from Jupyter via DAPR!", "transport": "dapr"}
)

await bus.publish(event)
print(f"📤 Published event: {event.id}")
print(f"   Type: {event.type}")
print(f"   Source: {event.source}")
print(f"   → Published via DAPR sidecar to {PUBSUB_NAME}/{PUBLISH_TOPIC}")

## 3. Inspect the Wire Format

Let's see exactly what goes on the wire. The `to_structured()` method produces
the canonical CloudEvent JSON — this is what both KNative and DAPR transports
put on the wire (in structured mode).

In [ ]:
import json

wire_event = CloudEvent(
    type="com.acme.order.created",
    source="/orders/api",
    data={"orderId": "12345", "amount": 99.99}
)

print("=== Wire format (identical for KNative and DAPR) ===")
print(json.dumps(wire_event.to_structured(), indent=2))

# Round-trip test
parsed = CloudEvent.from_structured(wire_event.to_structured())
assert parsed.type == wire_event.type
assert parsed.source == wire_event.source
assert parsed.data == wire_event.data
assert parsed.specversion == "1.0"
print("\n✅ Round-trip: structured → parse → identical CloudEvent")

## 4. Create an Event Stream Display

In [ ]:
from event_stream import EventStream

stream = EventStream(title="📡 DAPR Event Stream")
stream.display()

## 5. Register a Handler

Same `@bus.handler()` decorator as KNative — the handler doesn't know
or care which transport delivered the event.

When running in-cluster with DAPR sidecar, inbound events arrive via
`POST /events` (the route configured in `DaprSubscriptionConfig`).
The bus auto-detects DAPR vs binary vs structured format.

In [ ]:
@bus.handler("com.example.demo")
async def handle_demo(event: CloudEvent, ctx: MessageContext) -> Disposition:
    """Handle incoming demo events — same handler works for KNative and DAPR."""
    stream.append(event)
    return Disposition.COMPLETE

print("✅ Handler registered for: com.example.demo")
print("   Events will appear in the stream above ↑")

## 6. Local Dispatch (Test Without Sidecar)

`bus.dispatch()` invokes handlers locally — no DAPR sidecar needed.
Useful for testing handler logic.

In [ ]:
test_event = CloudEvent(
    type="com.example.demo",
    source="/demo/jupyter-dapr",
    data={"message": "Dispatched locally (no sidecar)!", "transport": "local"}
)

result = await bus.dispatch(test_event)
print(f"Handler returned: {result.value}")
print("Check the Event Stream above ↑")

## 7. Simulate DAPR Inbound Delivery

When the DAPR sidecar delivers an event, it POSTs to `/events` with either:
- A raw CloudEvent JSON (if publisher used `rawPayload=true`) — **our case**
- A DAPR envelope with `topic`, `pubsubname`, and `data`

The bus handles both automatically. Let's simulate both formats:

In [ ]:
# Simulate raw CloudEvent delivery (rawPayload=true)
raw_ce = CloudEvent(
    type="com.example.demo",
    source="/demo/jupyter-dapr",
    data={"message": "Raw CE via DAPR", "format": "structured"}
)
result = await bus.dispatch(raw_ce)
print(f"✅ Raw CE dispatch: {result.value}")

# Simulate DAPR-wrapped delivery (default mode)
wrapped_ce = CloudEvent.from_structured({
    "specversion": "1.0",
    "type": "com.example.demo",
    "source": "/demo/dapr-wrapped",
    "data": {"message": "DAPR-wrapped CE", "format": "wrapped"}
})
result = await bus.dispatch(wrapped_ce)
print(f"✅ Wrapped CE dispatch: {result.value}")
print("Both formats produce the same handler experience ↑")

## 8. Summary — KNative vs DAPR

| Aspect | KNative | DAPR |
|--------|---------|------|
| **Transport** | `KNativeTransport` | `DaprTransport` |
| **Publish via** | HTTP POST to Broker | HTTP POST to DAPR sidecar |
| **Subscribe via** | KNative Trigger → HTTP | DAPR subscription → HTTP |
| **Wire format** | Binary CE (headers) or Structured | Structured CE (`rawPayload=true`) |
| **Handler code** | `@bus.handler()` | `@bus.handler()` — **identical** |
| **CloudEvent on wire** | **Same** | **Same** |

### Switching transport — one line change:

```python
# KNative
bus.configure(KNativeTransport(broker_url="http://..."))

# DAPR
bus.configure(DaprTransport(
    pubsub_name="messaging",
    topic="knative-outbound",
    subscriptions=[DaprSubscription("messaging", "knative-inbound")],
))
```

Everything else stays the same.

In [ ]:
await bus.close()
print("🔒 Done")